# Import Dependencies

In [17]:
%load_ext autoreload
%autoreload 2
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import interpolate
import h5py
import pandas as pd
import kaleido # need pip install kaleido==0.1.0post1 for write_image to work
import re   
import os
import sys 
import time
import datetime

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# What experimental configuration do you want to use?

In [18]:
# Import the necessary class and functions from your module
from bender_functions import Bender
bender = Bender(config_module_name='jimenez_bender_config_A')
summary = bender.summary()
print(summary)

Bender initialized using: jimenez_bender_config_A.py
              BENDER SYSTEM SUMMARY               
Config:      jimenez_bender_config_A.py
Device:      Dev1
Motor Port:  port0
Direction:   POSITIVE = LEFT
--------------------------------------------------
Cal File:    FT56491.cal
Sample Rate: 1000.0 Hz
Ramp:        0.25 s
              BENDER SYSTEM SUMMARY               
Config:      jimenez_bender_config_A.py
Device:      Dev1
Motor Port:  port0
Direction:   POSITIVE = LEFT
--------------------------------------------------
Cal File:    FT56491.cal
Sample Rate: 1000.0 Hz
Ramp:        0.25 s


## Where do you want to save the data files?

In [20]:
# Define the base experiment name and folder
data_folder = r'C:\Users\jimen\desktop\BenderData'
base_name = '2026-03-13_CodeTest_'

# Create full file path
outputfile = bender.increment_file_name(f"{data_folder}\\{base_name}.h5")

# Mirror that name for the figure by simply swapping the extension
outputfig = outputfile.replace('.h5', '.png')

# Create a clean table view
print("\n" + "="*60)
print(f"{'FILE EXPORT SUMMARY':^60}")
print("-" * 60)
print(f"{'Type':<15} | {'Full System Path':<42}")
print("-" * 60)
print(f"{'HDF5 Data':<15} | {outputfile:<42}")
print(f"{'PNG Figure':<15} | {outputfig:<42}")
print("="*60 + "\n")


                    FILE EXPORT SUMMARY                     
------------------------------------------------------------
Type            | Full System Path                          
------------------------------------------------------------
HDF5 Data       | C:\Users\jimen\desktop\BenderData\2026-03-13_CodeTest_001.h5
PNG Figure      | C:\Users\jimen\desktop\BenderData\2026-03-13_CodeTest_001.png



## Biometrics: Input these before mounting

In [21]:
fishcode = "foam" 
segment = "anterior"
fishmass = 100    # Body mass in grams
fishlen_TL = 186     # Total length in mm
fishlen_SL = 185 # Standard length in mm

# Mount-specific dimensions: input these after mounting but BEFORE running the experiment
xsec_width = 25       # mm Cross sectional width of fish between the clamps at the axis of rotation
xsec_height = 83       # mm Cross sectional height of fish between the clamps at the axis of rotation

# Mounted biometrics
dbend = 123     # mm Distance from snout to the center of pressure ?
dclamp = 10     # mm Distance between the two clamps
dvert = 190         # mm Vertical distence from the transducer to the center of pressure
dhoriz = 10          # mm Horizontal distence from the transducer to the center of pressure

# What kind of experiment would you like to run?

If you want to drive curvature based on body thickness and red muscle strain, a simple combination can be performed to create inputs for all_curves.


In [22]:
test_type = "dynamic" # Select your test type!

# Bending motions
all_freqs = [.5,2]
all_curves = [1,2,3]
randomize = False 
cycles_per_step = 5    # How many times do you want to bend your specimen at each amplitude/frequency?
n_end_cycles = 2        # add cycles after last amplitude step
stim_cycles_in_step = np.array([2,3]) # This array defines which cycles to activate
# Bending motions for FREQUENCY SWEEP only (these will get ignore if you run a different test type)
duration = 60      # sec. How long to you want the whole test to last?
amplitude_frequency_exponent = 0   # should be between -1 and 0. Zero is constant amplitude, -1 is constant velocity, -0.5 is right in the middle.

# Electrical stiulation
is_stim = True # True stimulates the muscle. False is for passive tests
all_stimduties = [.1,.5]       # fractions of a cycle
all_stimphases = [-.1,0]      # fractions of a cycle
stim_pulse_rate = 75 # Hz shouldn't have to change this from 75
# Change these to whatever the stimulator panel is set to.
S1volts = 10 # Volts
S2volts = 10 # Volts
S1pulsedur = 2          # ms
S2pulsedur = 2          # ms

# Set only the "Dynamic" experiment parameters (the stuff that changes per fish)
bender.all_curves = all_curves
bender.all_freqs = all_freqs
bender.is_stim = is_stim
bender.all_stimphases = all_stimphases
bender.all_stimduties = all_stimduties

# Run the calculation method to generate the sequence attributes. This applies only to some test_types (dynamic, static)
if test_type in ['dynamic', 'static']:
    bender.organize_cycles(
        all_curves=all_curves,
        all_freqs=all_freqs,
        randomize=randomize,
        cycles_per_step=cycles_per_step,
        n_end_cycles=n_end_cycles,
        dclamp=dclamp,
        xsec_width=xsec_width,
        stim_cycles_in_step=stim_cycles_in_step,
        all_stimduties=all_stimduties,
        all_stimphases=all_stimphases,
        stim_pulse_rate=stim_pulse_rate
    )

    # Generate the numbers for the plot
    angle, anglevel, tnorm, freq, t = bender.make_dynamic_cycles(
        bender.period_by_cycle, 
        bender.freq_by_cycle, 
        bender.amp_by_cycle
    )
    bender.record_motor_signal(t, angle, anglevel, tnorm)

if test_type in ['sweep']:
    bender.duration = duration # duration of each sweep in seconds
    bender.all_freqs = all_freqs # frequencies of sweep in Hz (should be a list of 2 values: [start, end])
    bender.all_curves = all_curves # amplitude of sweep in degrees
    bender.xsec_width = xsec_width
    bender.amplitude_frequency_exponent = amplitude_frequency_exponent # exponent for how amplitude changes
   
    # Generate the numbers for the plot
    angle, anglevel, tnorm, freq, t = bender.make_frequency_sweep(
        bender.all_freqs, 
        bender.all_curves, 
        bender.amplitude_frequency_exponent, 
        bender.waitbefore
    )


# ADD STIMULI (code will ignore if is_stim is False)
# 1. Generate the stimulus signals
S1stimcmd, S2stimcmd = bender.make_stimuli()

# 2. Extract the Lonoff and Ronoff lists the function just saved
Lonoff = bender.Lonoff
Ronoff = bender.Ronoff



organize_cycles took 0.00022292137145996094 seconds


## Check experimental sequence

In [23]:
# CHECK CYCLE PARAMETERS
print("--- Generated Sequence Attributes ---")

pd.DataFrame({
    "freq (Hz)": bender.all_freqs,           # This is all_freqs_arr
    "curve (1/m)": bender.all_curves,       # This is all_curves_arr
    "amp (deg)": bender.all_degs,           # Expanded version of amps
    "strain (%)": bender.all_strains * 100,
    "strain rate (%/s)": bender.all_strainrates * 100,
    "duty (%)": bender.all_stimduties,      # Use the expanded array from the script
    "phase (%)": bender.all_stimphases      # Use the expanded array from the script
})

--- Generated Sequence Attributes ---


,freq (Hz),curve (1/m),amp (deg),strain (%),strain rate (%/s),duty (%),phase (%)
0,0.5,1,0.572958,1.25,3.926991,0.1,-0.1
1,0.5,1,0.572958,1.25,3.926991,0.1,0.0
2,0.5,1,0.572958,1.25,3.926991,0.5,-0.1
3,0.5,1,0.572958,1.25,3.926991,0.5,0.0
4,2.0,1,0.572958,1.25,15.707963,0.1,-0.1
5,2.0,1,0.572958,1.25,15.707963,0.1,0.0
6,2.0,1,0.572958,1.25,15.707963,0.5,-0.1
7,2.0,1,0.572958,1.25,15.707963,0.5,0.0
8,0.5,2,1.145916,2.50,7.853982,0.1,-0.1
9,0.5,2,1.145916,2.50,7.853982,0.1,0.0


## Input dimensions to calculate Moment of Inertia for Clamps and Specimen

In [24]:
clamp_offset = 20.0 # mm Distance between the rotating clamps front margin and the axis of rotation
front_h, front_w = 160, 30
back_h, back_w = 20, 10 
spec_length = 100.0

# Set the physics for the bender based on the mounting dimensions and specimen dimensions. This will be used to calculate the moment of inertia and other parameters for the bending
bender.set_physics(clamp_offset, front_h, front_w, back_h, back_w, spec_length)


           PHYSICS CONFIGURATION REPORT           
Mode                      | lateral              
Total Rotating MOI        | 1540379.44   | g*mm²
Lever Arm (r)             | 30.00        | mm
Specimen Mass             | 383.50       | g
--------------------------------------------------
TOTAL SYSTEM MASS         | 807.98       | g



## CALCULATE: important measurements to include in the file (H5) to be saved. 

In [ ]:
#Theoretically not necessary since most values are saved, but simplifies pipeline.  
test_section_pos = dbend/(fishlen_TL + dclamp + CLAMP_D)   # Position of the bending segment at the axis of rotation (fraction of total length). Assuming the clamp depth (CLAMP_D below) is 20. 

# Determine which cycles will have muscle stim

Six channels from the force transducer, plus the monitor channel from the S88 stimulator.

In [ ]:
SG0_chan = 'ai0'
SG1_chan = 'ai1'
SG2_chan = 'ai2'
SG3_chan = 'ai3'
SG4_chan = 'ai4'
SG5_chan = 'ai5'

stim_monitor_chan = 'ai6'

inchannels = [SG0_chan, SG1_chan, SG2_chan, SG3_chan, SG4_chan, SG5_chan,
                stim_monitor_chan]
inchannel_names = ['SG0', 'SG1', 'SG2', 'SG3', 'SG4', 'SG5',
                    'stim_monitor']

bender.set_input_channels(inchannels, inchannel_names)

# Plot angles and pulses to CHECK

In [ ]:
fig = make_subplots(rows = 2, cols = 1,
                   shared_xaxes=True)
fig.add_trace(
    go.Scatter(x = tnorm, y = angle, mode="lines", name="angle"),
    row=1, col=1)

for onoff in Lonoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], fillcolor="black", opacity=0.25, line_width=0,
                      row=1, col=1)

for onoff in Ronoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], opacity=0.7, line_width=1,
                      row=1, col=1)

fig.update_yaxes(title_text = "angle (deg)", row=1)
fig.add_trace(
    go.Scatter(x = tnorm, y = anglevel, mode="lines", name="anglevel"),
    row=2, col=1)

fig.update_yaxes(title_text = "angular velocity (deg/s)", row=2)
fig.update_xaxes(title_text = "time (s)", row=2)

# START BENDING!! 

This is the main code block that runs the experiment. It sets up the DAQ, sends the output, records the input, and writes it to the file.

In [ ]:
aidata = bender.run(device_name)

In [ ]:
forcetorque = bender.applyCalibration(aidata)
forcetorque_names = ['xForce', 'yForce', 'zForce', 'xTorque', 'yTorque', 'zTorque']

In [ ]:
angle_measured = bender.angle # Need to make sure to add angles for the internal motor encoder, the command angle, and the measured angle from the external encoder

In [ ]:
# Save the actual text of your config file for ultimate traceability
with h5py.File(outputfile, 'w') as f:

    with open(f"{bender.config_name}.py", 'r') as cfg_file:
        f.attrs['Config_File_Content'] = cfg_file.read()
        f.attrs['Config_Module_Used'] = bender.config_name

    # --- Group 1: General Experiment Information ---
    g_info = f.create_group('ExperimentInfo')
    g_info.attrs['EndTime'] = bender.endTime.strftime('%Y-%m-%d %H:%M:%S %Z')
    g_info.attrs['FishCode'] = fishcode
    g_info.attrs['Segment'] = segment

    # --- Group 2: Specimen Geometry and Setup ---
    g_specimen = f.create_group('Biometrics')
    g_specimen.attrs['FishLength_mm'] = fishlen
    g_specimen.attrs['FishMass_g'] = fishmass
    g_specimen.attrs['FishCrossSectionWidth_mm'] = xsec_width
    g_specimen.attrs['FishCrossSectionHeight_mm'] = xsec_height
    g_specimen.attrs['TestSectionPosition_perc'] = test_section_pos

    # --- Group 3: Mount Geometry ---
    g_mount = f.create_group('MountGeometry')
    g_mount.attrs['BendLocation_mm'] = dbend
    g_mount.attrs['ClampDistance_mm'] = dclamp
    g_mount.attrs['DistanceFromTransducerVert_mm'] = dvert
    g_mount.attrs['DistanceFromTransducerHoriz_mm'] = dhoriz

    # --- Group 4: Inertial/Mass Properties (MOI) ---
    g_moi = f.create_group('InertialProperties')
    g_moi.attrs['TotalSystemMass_g'] = Total_Mass_System
    g_moi.attrs['TotalSystemMOI_gmm2'] = I_total_system

    # --- Group 5: Physical Dimensions used for calculations ---
    g_dims = f.create_group('CalculationDimensionsMOI')
    g_dims.attrs['Clamp_Height_mm'] = CLAMP_H
    g_dims.attrs['Clamp_Width_mm'] = CLAMP_W
    g_dims.attrs['Clamp_Depth_mm'] = CLAMP_D
    g_dims.attrs['Clamp_Density_gmm3'] = RHO_CLAMP
    g_dims.attrs['Specimen_Height_mm'] = SPECIMEN_H
    g_dims.attrs['Specimen_Depth_mm'] = SPECIMEN_D
    g_dims.attrs['Specimen_FrontHeight_mm'] = FRONT_H
    g_dims.attrs['Specimen_FrontWidth_mm'] = FRONT_W
    g_dims.attrs['Specimen_BackHeight_mm'] = BACK_H
    g_dims.attrs['Specimen_BackWidth_mm'] = BACK_W
    g_dims.attrs['Specimen_Density_gmm3'] = RHO_OBJECT 

    # Start saving raw data, calibrated data, and output data
    gin = f.create_group('RawInput')
    gin.attrs['SampleFrequency'] = samplefreq
  
    # Store measurements
    gin.create_dataset('forcetransducer', data=aidata[:6,:])
    gin.create_dataset('Stimulation_monitor', data=aidata[6,:])

    gcal = f.create_group('Calibrated')
    for ft1, name1 in zip(forcetorque, forcetorque_names):
        gcal.create_dataset(name1, data=ft1)
    gcal.create_dataset('CalibrationMatrix', data=bender.calibration) # Save the calibration matrix used

    ds = gcal.create_dataset('Encoder', data=bender.angledata) # DOUBLE CHECK THIS
    ds.attrs['CountsPerRev'] = encoder_counts_per_rev

    # save the output data
    gout = f.create_group('Output')
    gout.attrs['SampleFrequency'] = outputfreq
    gout.create_dataset('DigitalOut', data=dig)
    gout.create_dataset('SyncInTrainDur', data=S1actcmd)
    gout.create_dataset('SyncInS2Del', data=S2actcmd)
    gout.attrs['S1side'] = S1side
    gout.attrs['S2side'] = S2side
    gout.attrs['S1volts'] = S1volts
    gout.attrs['S2volts'] = S2volts
    gout.attrs['S1pulsedur_ms'] = S1pulsedur    
    gout.attrs['S2pulsedur_ms'] = S2pulsedur    
    
    # Save stimulus parameters
    gout = f.create_group('NominalStimulus')
    gout.attrs['Type'] = 'Dynamic'

    gout.create_dataset('t', data=t)
    ds = gout.create_dataset('Position', data=angle)
    ds.attrs['Units'] = 'deg'
    ds = gout.create_dataset('Velocity', data=anglevel)
    ds.attrs['Units'] = 'deg/sec'
    gout.create_dataset('tnorm', data=tnorm)
    gout.create_dataset('Lonoff', data=Lonoff)
    gout.create_dataset('Ronoff', data=Ronoff)

    # Save bending parameters
    # Think about how to save all amps, curves, strains, velocities, etc.
    all_amps = []  # TODO: Replace with actual amplitude data if available
    gout.attrs['Curvatures'] = all_curves
    gout.attrs['Frequencies'] = all_freqs
    gout.attrs['CyclesPerStep'] = cycles_per_step
    gout.attrs['EndCycles'] = n_end_cycles
    gout.attrs['FrequencyByCycle'] = freq_by_cycle
    gout.attrs['AmplitudeByCycle'] = amp_by_cycle
    gout.attrs['IsStimByCycle'] = is_stim_cycle
    gout.attrs['CycleRandomOrder'] = order
    gout.attrs['MovementDuration'] = movedur

    # Add estimated red muscle strain and strain rate
    gout.attrs['Strains'] = allstrains
    gout.attrs['StrainRates'] = allstrainrates  

    gout.attrs['WaitPre'] = waitbefore
    gout.attrs['WaitPost'] = waitafter
    gout.attrs['PrePostStimDur'] = prepoststim_dur
    gout.attrs['ScaleFactor'] = scale
    gout.attrs['PositiveMotorDirection'] = positive_motor_direction

    gout.attrs['StimulationOn'] = is_stim
    gout.attrs['StimulationDuty'] = all_stimduties
    gout.attrs['StimulationPhase'] = all_stimphases
    gout.attrs['StimulationPulseRate'] = stim_pulse_rate


##  Zero the data for plotting results

In [ ]:
for ft1 in forcetorque:
    ft1 -= np.mean(ft1[t < 0])

## QUALITY CONTROL: Plot control signals AND raw torque over time

In [ ]:
# Make sure to plot the correct bending axis. It depends on sensor mounting and orientation!
fig = make_subplots(rows = 4, cols = 1,
                   shared_xaxes=True)
#fig.add_trace(
 #   go.Scatter(x = tnorm, y = angle_measured, mode="lines", name="angle_enc"),
  #  row=1, col=1)
fig.add_trace(
    go.Scatter(x = tnorm, y = angle, mode="lines", name="angle_cmd"),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x = tnorm, y = aidata[6,:], mode="lines", name="stim"),
    row=4, col=1)
fig.add_trace(
    go.Scatter(x = tnorm, y = forcetorque[3,:], mode="lines", name="Tx"),
    row=2, col=1)
# fig.add_trace(
#     go.Scatter(x = tnorm, y = forcetorque[1,:], mode="lines", name="Fy"),
#     row=4, col=1)
fig.add_trace(
    go.Scatter(x = tnorm, y = forcetorque[5,:], mode="lines", name="Tz"),
    row=3, col=1)

for onoff in Lonoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], fillcolor="black", opacity=0.25, line_width=0,
                        row="all", col="all")

for onoff in Ronoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], opacity=0.7, line_width=1,
                      row="all", col="all")
fig.update_yaxes(title_text = "angle (deg)", row=1)
fig.update_yaxes(title_text = "Tx (Nm)", row=2)
fig.update_yaxes(title_text = "Tz (Nm)", row=3)
fig.update_yaxes(title_text = "stim", row=4)
fig.update_xaxes(title_text = "time (s)", row=3)
fig.update_layout(title_text = bender.filename)

# Save the plot in output folder
fig.write_image(outputfig, format= 'png', width=1200, height=800)


## QUALITY CONTROL: Do the loops like nice?

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=angle, y=forcetorque[3,:]))
fig.update_yaxes(title_text="torque (Nm)")
fig.update_xaxes(title_text="angle (deg)")